<div align="center">

# Day 03: Policy Gradient Methods

**Reinforcement Learning and Language Model Alignment (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-03) · [Lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html) · [Interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lab.html) · [PDF notes](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/Day03_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this notebook is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from itertools import product
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.4), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 9})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


print('environment ready')

## Python step 3: Vectors, the softmax and learning curves

A policy over a few actions is a vector of preferences turned into probabilities by the softmax. This step writes the softmax and the gradient of a log-probability with NumPy, runs REINFORCE on a three-armed bandit with and without a baseline and draws the two learning curves &#91;[6](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#ref-6), [7](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#ref-7)&#93;.

### Preferences and the softmax

In [ ]:
import numpy as np
def softmax(z):
    """Probabilities from preferences; subtracting the maximum avoids overflow."""
    e = np.exp(z - z.max())
    return e / e.sum()

theta = np.array([0.0, 0.0, 0.0])        # one preference per arm
p = softmax(theta)
print(p, p.sum())
print(softmax(np.array([1.0, 2.0, 3.0])).round(3))

The softmax exponentiates every preference and divides by the sum, so the result is positive and sums to 1. Subtracting the largest preference first does not change the result and prevents overflow for large values, a standard precaution.

### The gradient of a log-probability

In [ ]:
a = 1
grad_logp = np.eye(3)[a] - p              # gradient of log pi(a) for a softmax policy
print("log pi(a):", round(float(np.log(p[a])), 4))
print("gradient :", grad_logp.round(3), " sum:", round(float(grad_logp.sum()), 10))

For a softmax policy, the gradient of the log-probability of action $a$ with respect to the preferences is the one-hot vector of $a$ minus the probabilities. `np.eye(3)[a]` takes row `a` of the identity matrix, which is that one-hot vector. The components always sum to zero.

### REINFORCE on a bandit, with and without a baseline

In [ ]:
win = np.array([0.2, 0.8, 0.5])          # true win rates, unknown to the agent
lr = 0.1
def reinforce(use_baseline, seed=0, steps=500):
    rng = np.random.default_rng(seed)
    th, avg, curve = np.zeros(3), 0.0, []
    for t in range(steps):
        p = softmax(th)
        a = rng.choice(3, p=p)
        r = float(rng.random() < win[a])
        adv = r - avg if use_baseline else r
        th += lr * adv * (np.eye(3)[a] - p)
        avg += 0.05 * (r - avg)
        curve.append(p[1])
    return np.array(curve)

plain, based = reinforce(False), reinforce(True)
print("probability of the best arm after 500 steps:", round(float(plain[-1]), 3), "and", round(float(based[-1]), 3))

`rng.choice(3, p=p)` draws an arm with the probabilities of the policy. The update adds the learning rate times the advantage times the gradient of the log-probability. With the baseline, the advantage is the reward minus a running average, so an ordinary reward no longer pushes the chosen arm up.

### A learning curve

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.2))
plt.plot(plain, label="no baseline")
plt.plot(based, label="running-average baseline")
plt.xlabel("step"); plt.ylabel("probability of the best arm"); plt.legend()
plt.show()

Each call of `plt.plot` adds one line, and `label` names it for the legend. A learning curve over one seed is an illustration, not a result; the notebook repeats such curves over several seeds, as the rules of Day 1 require.

**Quick check.** Why does subtracting a baseline that does not depend on the action leave the gradient unbiased?

<details><summary>Answer</summary>

Because the expected value of the gradient of the log-probability is zero: The probabilities sum to 1 for every parameter value, so their gradients sum to zero, and a constant times zero is zero.

</details>

### Exercises for Python step 3

**Exercise 3.1.** Store the largest probability of `softmax(np.array([1.0, 2.0, 3.0]))`, rounded to three decimals, in `p_max`.

In [ ]:
p_max = None   # your answer

check("Exercise 3.1", p_max, 0.665)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_max = round(float(softmax(np.array([1.0, 2.0, 3.0])).max()), 3)

check("Exercise 3.1", p_max, 0.665)

**Exercise 3.2.** The entropy of a distribution is `-(p * np.log(p)).sum()`. Store the entropy of the uniform policy over three arms, rounded to four decimals, in `h_uniform`.

In [ ]:
h_uniform = None   # your answer

check("Exercise 3.2", h_uniform, 1.0986)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pu = softmax(np.zeros(3))
h_uniform = round(float(-(pu * np.log(pu)).sum()), 4)

check("Exercise 3.2", h_uniform, 1.0986)

**Exercise 3.3.** Compute the gradient of the log-probability of arm 0 under the uniform policy and store the sum of its components in `grad_sum`.

In [ ]:
grad_sum = None   # your answer

check("Exercise 3.3", grad_sum, 0.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pu = softmax(np.zeros(3))
grad_sum = float((np.eye(3)[0] - pu).sum())

check("Exercise 3.3", grad_sum, 0.0)

## 1. TokenWorld and its exact optimum

The environment of Day 1 and the softmax policy over its 161 states.

In [ ]:
# TokenWorld: sequence generation as a Markov decision process.
#
#   state     s_t = the tokens generated so far, the prefix
#   action    a_t = the next token, from the vocabulary
#   transition      s_{t+1} = s_t followed by a_t, deterministic
#   reward    r_t = 0 until the sequence ends, then a programmatic score
#   done      when <eos> is emitted or the length cap is reached
#
# The tiny version below has four tokens and four positions, so the whole MDP can be
# enumerated and solved exactly.

TINY_VOCAB = ["the", "cat", "sat", "<eos>"]     # 4 tokens
EOS = 3
MAXLEN = 4

def score_sequence(seq, vocab=TINY_VOCAB):
    """The true reward: hand-written, deterministic, known to the designer and hidden from the agent.

    A well-formed utterance in this toy language: starts with 'the', contains 'cat' exactly
    once, ends with '<eos>', and is short. Everything else is penalised.
    """
    toks = [vocab[i] for i in seq if vocab[i] != "<eos>"]
    r = 0.0
    if not toks:
        return -1.0                                    # empty utterance
    if toks[0] == "the":     r += 1.0
    else:                    r -= 0.5
    n_cat = toks.count("cat")
    if n_cat == 1:           r += 1.5
    elif n_cat == 0:         r -= 0.5
    else:                    r -= 0.8 * (n_cat - 1)     # repetition is punished
    if "sat" in toks and "cat" in toks and toks.index("cat") < toks.index("sat"):
        r += 1.0                                       # word order matters
    r -= 0.25 * len(toks)                              # brevity bonus
    return float(r)


def enumerate_states(vocab_n=4, maxlen=MAXLEN):
    """Every reachable prefix. A prefix ending in EOS is terminal."""
    states = [()]
    frontier = [()]
    for _ in range(maxlen):
        nxt = []
        for s in frontier:
            if s and s[-1] == EOS:
                continue
            for a in range(vocab_n):
                nxt.append(s + (a,))
        states += nxt
        frontier = nxt
    return states


def is_terminal(s, maxlen=MAXLEN):
    return len(s) >= maxlen or (len(s) > 0 and s[-1] == EOS)


def step(s, a, maxlen=MAXLEN, vocab=TINY_VOCAB):
    """One transition. The reward is zero until the episode ends."""
    s2 = s + (a,)
    done = is_terminal(s2, maxlen)
    r = score_sequence(s2, vocab) if done else 0.0
    return s2, r, done


def rollout(policy_fn, rng, maxlen=MAXLEN, vocab_n=4):
    """Run one episode. `policy_fn(state) -> action probabilities`."""
    s, traj = (), []
    while not is_terminal(s, maxlen):
        p = policy_fn(s)
        a = int(rng.choice(len(p), p=p))
        s2, r, done = step(s, a, maxlen)
        traj.append((s, a, r))
        s = s2
    return traj, s

In [ ]:
S = enumerate_states()
NONTERM = [s for s in S if not is_terminal(s)]
NA = len(TINY_VOCAB)
IDX = {s: i for i, s in enumerate(NONTERM)}

V_STAR = {s: 0.0 for s in S}
for _ in range(MAXLEN + 2):
    for s in S:
        if is_terminal(s):
            continue
        V_STAR[s] = max(step(s, a)[1] + (0.0 if step(s, a)[2] else V_STAR[step(s, a)[0]])
                        for a in range(NA))
OPT = V_STAR[()]
print(f"exact optimum carried from Day 1: {OPT:.4f}")

def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

def make_policy(theta):
    P = softmax(theta)
    return lambda s: P[IDX[s]]

def expected_return(theta, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    pol = make_policy(theta)
    return float(np.mean([rollout(pol, rng)[0][-1][2] for _ in range(n)]))

def visitation(theta):
    """How often the policy actually reaches each state. Deterministic transitions, so exact."""
    P = softmax(theta); d = np.zeros(len(NONTERM)); front = {(): 1.0}
    while front:
        nxt = {}
        for s, w in front.items():
            d[IDX[s]] += w
            for a in range(NA):
                s2 = s + (a,)
                if not is_terminal(s2):
                    nxt[s2] = nxt.get(s2, 0.0) + w * P[IDX[s], a]
        front = nxt
    return d / d.sum()

def policy_entropy(theta, weighted=True):
    """Entropy AVERAGED OVER THE STATES THE POLICY ACTUALLY VISITS.

    Averaging uniformly over the whole table is the standard mistake: most of the 161 states are
    never reached, their logits never move, and their untouched uniform entropy masks a total
    collapse in the handful of states that matter."""
    P = softmax(theta)
    H = -(P * np.log(P + 1e-12)).sum(1)
    w = visitation(theta) if weighted else np.ones(len(NONTERM)) / len(NONTERM)
    return float((H * w).sum())

## 2. The variance of the REINFORCE estimator &#91;[1](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#ref-1)&#93;

*Lecture: [Variance and baselines](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#variance-and-baselines)*

**Python move: estimating a variance.** The gradient is estimated sixty times at the same parameters, and the variance over the repetitions is averaged over all components.

In [ ]:
def reinforce_grad(theta, batch, rng, baseline=None, use_rtg=True):
    """Returns the gradient estimate and the mean return of the batch."""
    P = softmax(theta)
    pol = lambda s: P[IDX[s]]
    grad = np.zeros_like(theta); returns = []
    for _ in range(batch):
        traj, _ = rollout(pol, rng)
        G_total = traj[-1][2]
        returns.append(G_total)
        G = 0.0
        for t in range(len(traj) - 1, -1, -1):
            s, a, r = traj[t]
            G = r + G
            adv = (G if use_rtg else G_total) - (baseline[IDX[s]] if baseline is not None else 0.0)
            onehot = np.zeros(NA); onehot[a] = 1.0
            grad[IDX[s]] += adv * (onehot - P[IDX[s]])
    return grad / batch, float(np.mean(returns))

# How noisy is it? Measure the variance of the estimator at a fixed theta.
rng = np.random.default_rng(0)
theta0 = np.zeros((len(NONTERM), NA))
V_hat = {s: 0.0 for s in S}                     # a value baseline, fitted below
for _ in range(200):
    traj, _ = rollout(make_policy(theta0), rng)
    G = 0.0
    for s, a, r in reversed(traj):
        G = r + G; V_hat[s] += 0.05 * (G - V_hat[s])
base = np.array([V_hat[s] for s in NONTERM])

rows = []
for label, kw in (("total return, no baseline", dict(use_rtg=False)),
                  ("reward-to-go", dict(use_rtg=True)),
                  ("reward-to-go + baseline", dict(use_rtg=True, baseline=base))):
    for B in (1, 8, 64):
        gs = np.array([reinforce_grad(theta0, B, np.random.default_rng(100 + i), **kw)[0].ravel()
                       for i in range(60)])
        rows.append(dict(estimator=label, batch=B, grad_variance=float(gs.var(0).mean())))
var = pd.DataFrame(rows)
print("VARIANCE of the gradient estimator on TokenWorld (lower is better)\n")
print(var.pivot(index="estimator", columns="batch", values="grad_variance").round(6).to_string())
b64 = var[var.batch == 64].set_index("estimator").grad_variance
print(f"\nreward-to-go vs total return : {b64['total return, no baseline'] / b64['reward-to-go']:.2f}x")
print(f"baseline vs no baseline      : "
      f"{b64['reward-to-go'] / b64['reward-to-go + baseline']:.2f}x")

**Exercise A.** By what factor does the baseline reduce the variance at batch 64? Store `b64['reward-to-go'] / b64['reward-to-go + baseline']` in `base_gain`.

In [ ]:
base_gain = None   # your answer

check("Exercise A", base_gain, float(b64['reward-to-go'] / b64['reward-to-go + baseline']))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
base_gain = float(b64['reward-to-go'] / b64['reward-to-go + baseline'])

check("Exercise A", base_gain, float(b64['reward-to-go'] / b64['reward-to-go + baseline']))

## 3. REINFORCE, trained

*Lecture: [Actor-critic](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#actor-critic)*

In [ ]:
def train_reinforce(n_iter=400, batch=32, lr=0.35, use_rtg=True, use_baseline=True,
                    entropy_coef=0.0, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((len(NONTERM), NA))
    Vb = np.zeros(len(NONTERM))
    hist = []
    for it in range(n_iter):
        g, mret = reinforce_grad(theta, batch, rng, baseline=(Vb if use_baseline else None),
                                 use_rtg=use_rtg)
        if entropy_coef:
            P = softmax(theta)
            g += entropy_coef * (-P * (np.log(P + 1e-12) + 1))
        theta += lr * g
        if use_baseline:                                   # fit the baseline on fresh returns
            pol = make_policy(theta)
            for _ in range(batch // 4):
                traj, _ = rollout(pol, rng)
                G = 0.0
                for s, a, r in reversed(traj):
                    G = r + G; Vb[IDX[s]] += 0.15 * (G - Vb[IDX[s]])
        if it % 10 == 0:
            hist.append((it, mret, policy_entropy(theta)))
    return theta, pd.DataFrame(hist, columns=["iter", "return", "entropy"])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
summary = []
for label, kw, col in (("total return, no baseline", dict(use_rtg=False, use_baseline=False), "#c0392b"),
                       ("reward-to-go", dict(use_rtg=True, use_baseline=False), "#e67e22"),
                       ("reward-to-go + baseline", dict(use_rtg=True, use_baseline=True), "#27ae60")):
    runs = [train_reinforce(seed=s, **kw) for s in range(8)]
    R = np.array([r[1]["return"].values for r in runs])
    E = np.array([r[1]["entropy"].values for r in runs])
    x = runs[0][1]["iter"].values
    ax[0].plot(x, R.mean(0), color=col, label=label)
    ax[0].fill_between(x, np.percentile(R, 10, 0), np.percentile(R, 90, 0), color=col, alpha=.15)
    ax[1].plot(x, E.mean(0), color=col, label=label)
    summary.append(dict(estimator=label, final_return=R[:, -1].mean(),
                        pct_of_optimum=100 * R[:, -1].mean() / OPT,
                        sd_across_seeds=R[:, -1].std(), final_entropy=E[:, -1].mean()))
ax[0].axhline(OPT, ls="--", lw=1, color="#111111", label="exact optimum")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("mean return"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("policy entropy (nats)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(pd.DataFrame(summary).round(3).to_string(index=False))

## 4. Actor-critic with generalised advantage estimation &#91;[3](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#ref-3), [4](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#ref-4)&#93;

*Lecture: [Actor-critic](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#actor-critic)*

In [ ]:
def train_actor_critic(n_iter=400, batch=32, lr=0.35, lr_v=0.25, lam=0.95, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((len(NONTERM), NA)); V = np.zeros(len(NONTERM))
    hist = []
    for it in range(n_iter):
        P = softmax(theta); pol = lambda s: P[IDX[s]]
        g = np.zeros_like(theta); rets = []
        for _ in range(batch):
            traj, _ = rollout(pol, rng)
            rets.append(traj[-1][2])
            # GAE
            deltas, states, acts = [], [], []
            for t, (s, a, r) in enumerate(traj):
                s2 = s + (a,)
                v2 = 0.0 if is_terminal(s2) else V[IDX[s2]]
                deltas.append(r + v2 - V[IDX[s]]); states.append(s); acts.append(a)
            A = np.zeros(len(deltas)); run = 0.0
            for t in range(len(deltas) - 1, -1, -1):
                run = deltas[t] + lam * run; A[t] = run
            for t, (s, a) in enumerate(zip(states, acts)):
                oh = np.zeros(NA); oh[a] = 1.0
                g[IDX[s]] += A[t] * (oh - P[IDX[s]])
                V[IDX[s]] += lr_v * deltas[t]
        theta += lr * g / batch
        if it % 10 == 0:
            hist.append((it, float(np.mean(rets)), policy_entropy(theta)))
    return theta, pd.DataFrame(hist, columns=["iter", "return", "entropy"])

rows = []
fig, ax = plt.subplots(figsize=(7, 3.3))
for lam, col in ((0.0, "#8e44ad"), (0.5, "#2980b9"), (0.95, "#27ae60"), (1.0, "#e67e22")):
    runs = [train_actor_critic(lam=lam, seed=s) for s in range(6)]
    R = np.array([r[1]["return"].values for r in runs])
    x = runs[0][1]["iter"].values
    ax.plot(x, R.mean(0), color=col, label=f"GAE lambda = {lam}")
    rows.append(dict(lam=lam, final_return=R[:, -1].mean(), sd=R[:, -1].std(),
                     iters_to_90pct=int(np.argmax(R.mean(0) > 0.9 * OPT) * 10)
                     if (R.mean(0) > 0.9 * OPT).any() else -1))
ax.axhline(OPT, ls="--", lw=1, color="#111111", label="optimum")
ax.set_xlabel("iteration"); ax.set_ylabel("mean return"); ax.legend(fontsize=8)
ax.set_title("Actor-critic: the bias-variance dial")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 5. Plain policy gradient against PPO, under the same data reuse &#91;[5](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#ref-5)&#93;

*Lecture: [Trust regions and PPO](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#trust-regions-and-ppo)*

In [ ]:
def train_pg(n_iter=300, batch=32, lr=0.35, clip=None, epochs=4, entropy_coef=0.0, seed=0):
    """clip=None -> vanilla policy gradient. clip=eps -> PPO with `epochs` reuse passes."""
    rng = np.random.default_rng(seed)
    theta = np.zeros((len(NONTERM), NA)); V = np.zeros(len(NONTERM))
    hist = []
    for it in range(n_iter):
        P_old = softmax(theta); pol = lambda s: P_old[IDX[s]]
        data, rets = [], []
        for _ in range(batch):
            traj, _ = rollout(pol, rng)
            rets.append(traj[-1][2])
            deltas = []
            for (s, a, r) in traj:
                s2 = s + (a,)
                v2 = 0.0 if is_terminal(s2) else V[IDX[s2]]
                deltas.append(r + v2 - V[IDX[s]])
            A = np.zeros(len(deltas)); run = 0.0
            for t in range(len(deltas) - 1, -1, -1):
                run = deltas[t] + 0.95 * run; A[t] = run
            for t, (s, a, r) in enumerate(traj):
                data.append((IDX[s], a, A[t]))
                V[IDX[s]] += 0.25 * deltas[t]
        A_all = np.array([d[2] for d in data])
        A_all = (A_all - A_all.mean()) / (A_all.std() + 1e-8)

        n_pass = epochs
        for _ in range(n_pass):
            P = softmax(theta); g = np.zeros_like(theta)
            for (si, a, _), adv in zip(data, A_all):
                ratio = P[si, a] / (P_old[si, a] + 1e-12)
                oh = np.zeros(NA); oh[a] = 1.0
                if clip is None:
                    coef = adv
                else:
                    if (adv >= 0 and ratio > 1 + clip) or (adv < 0 and ratio < 1 - clip):
                        continue                      # outside the trust band: no gradient
                    coef = ratio * adv
                g[si] += coef * (oh - P[si])
            if entropy_coef:
                g += entropy_coef * (-P * (np.log(P + 1e-12) + 1))
            theta += lr * g / len(data)
        kl = float(np.mean((softmax(theta) * (np.log(softmax(theta) + 1e-12)
                                              - np.log(P_old + 1e-12))).sum(1)))
        if it % 10 == 0:
            hist.append((it, float(np.mean(rets)), policy_entropy(theta), kl))
    return theta, pd.DataFrame(hist, columns=["iter", "return", "entropy", "kl"])

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
rows = []
LR = 20.0
for label, kw, col in (("1 pass, no clip, lr 1", dict(lr=1.0, clip=None, epochs=1), "#2980b9"),
                       (f"8 passes, NO clip, lr {LR:.0f}", dict(lr=LR, clip=None, epochs=8), "#c0392b"),
                       (f"8 passes, PPO clip, lr {LR:.0f}", dict(lr=LR, clip=0.2, epochs=8), "#27ae60")):
    runs = [train_pg(seed=s, **kw) for s in range(6)]
    R = np.array([r[1]["return"].values for r in runs])
    K = np.array([r[1]["kl"].values for r in runs])
    E = np.array([r[1]["entropy"].values for r in runs])
    x = runs[0][1]["iter"].values
    ax[0].plot(x, R.mean(0), color=col, label=label)
    ax[0].fill_between(x, np.percentile(R, 10, 0), np.percentile(R, 90, 0), color=col, alpha=.12)
    ax[1].semilogy(x, np.maximum(K.mean(0), 1e-8), color=col, label=label)
    ax[2].plot(x, E.mean(0), color=col, label=label)
    rows.append(dict(config=label, final_return=R[:, -1].mean(),
                     pct_of_optimum=100 * R[:, -1].mean() / OPT,
                     worst_seed=R[:, -1].min(), mean_KL_per_update=K.mean()))
ax[0].axhline(OPT, ls="--", lw=1, color="#111111")
ax[0].set_title("return"); ax[1].set_title("KL between consecutive policies")
ax[2].set_title("policy entropy")
for a_ in ax:
    a_.set_xlabel("iteration"); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Exercise B.** What percentage of the optimum does plain policy gradient with eight passes at a learning rate of 20 reach? Store `rows[1]["pct_of_optimum"]` in `pct_plain`.

In [ ]:
pct_plain = None   # your answer

check("Exercise B", pct_plain, float(rows[1]["pct_of_optimum"]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pct_plain = float(rows[1]["pct_of_optimum"])

check("Exercise B", pct_plain, float(rows[1]["pct_of_optimum"]))

## 6. Entropy over the visited states

*Lecture: [Entropy](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#entropy)*

In [ ]:
rows = []
for beta in (0.0, 0.005, 0.02, 0.05, 0.15):
    runs = [train_pg(lr=1.0, clip=0.2, epochs=8, entropy_coef=beta, seed=s) for s in range(6)]
    R = np.array([r[1]["return"].values for r in runs])
    E = np.array([r[1]["entropy"].values for r in runs])
    theta = runs[0][0]
    P = softmax(theta)
    rows.append(dict(entropy_coef=beta, final_return=R[:, -1].mean(),
                     pct_of_optimum=100 * R[:, -1].mean() / OPT,
                     final_entropy=E[:, -1].mean(),
                     entropy_all_states=policy_entropy(theta, weighted=False),
                     max_action_prob=float((P.max(1) * visitation(theta)).sum())))
ent = pd.DataFrame(rows)
print(ent.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(ent.entropy_coef, ent.pct_of_optimum, "o-", color="#2980b9", label="% of optimum")
ax2 = ax.twinx()
ax2.plot(ent.entropy_coef, ent.final_entropy, "s--", color="#e67e22", label="final entropy")
ax.set_xlabel("entropy coefficient beta"); ax.set_ylabel("% of optimum", color="#2980b9")
ax2.set_ylabel("policy entropy (nats)", color="#e67e22")
ax.set_title("Return and entropy pull in opposite directions")
plt.tight_layout(); plt.show()

print("\nMaximum entropy for 4 actions is ln(4) = %.3f nats." % np.log(4))
print("the policy visits.")

**Exercise C.** Store the visitation-weighted final entropy at an entropy bonus of 0 from the table `ent` in `h_collapsed`.

In [ ]:
h_collapsed = None   # your answer

check("Exercise C", h_collapsed, float(ent.loc[ent.entropy_coef == 0, "final_entropy"].iloc[0]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
h_collapsed = float(ent.loc[ent.entropy_coef == 0, "final_entropy"].iloc[0])

check("Exercise C", h_collapsed, float(ent.loc[ent.entropy_coef == 0, "final_entropy"].iloc[0]))

## 7. Your field: three policy gradient learners

*Lecture: [Your field](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#your-field)*

Change `DOMAIN` to `drone` or `evacuation` and run the cell again. The cell takes up to half a minute.

In [ ]:
DOMAIN = "warehouse"     # choose: warehouse, drone or evacuation

def make_domain(name):
    """A grid with walls, hazards, a start, a goal and a chance that the agent slips in a random direction."""
    if name == "warehouse":     # shelves are walls; a forklift lane is dangerous
        return dict(rows=6, cols=8, walls={(1, 2), (2, 2), (3, 2), (1, 5), (2, 5), (3, 5), (4, 5)}, hazards={(5, 3), (5, 4), (4, 3)},
                    start=(5, 0), goal=(0, 7), slip=0.05, penalty=-20.0)
    if name == "drone":         # wind and a no-fly zone near the ground
        return dict(rows=5, cols=8, walls={(0, 3), (1, 3)}, hazards={(4, c) for c in range(1, 7)},
                    start=(3, 0), goal=(3, 7), slip=0.20, penalty=-50.0)
    if name == "evacuation":    # walls, fire and smoke
        return dict(rows=6, cols=6, walls={(2, 1), (2, 2), (2, 3), (4, 3), (4, 4)}, hazards={(3, 4), (1, 4), (3, 1)},
                    start=(5, 0), goal=(0, 5), slip=0.10, penalty=-100.0)
    raise ValueError(name)

MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def domain_step(D, s, a, rng):
    """One move; a slip replaces the action by a random one, walls block, a hazard costs its penalty and resets."""
    if rng.random() < D["slip"]:
        a = int(rng.integers(4))
    r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
    if not (0 <= r < D["rows"] and 0 <= c < D["cols"]) or (r, c) in D["walls"]:
        r, c = s
    if (r, c) in D["hazards"]:
        return D["start"], D["penalty"], False
    if (r, c) == D["goal"]:
        return (r, c), 0.0, True
    return (r, c), -1.0, False

def greedy_path(D, table, n=40):
    """The path that follows the best action from the start, without slips."""
    s, path = D["start"], [D["start"]]
    for _ in range(n):
        a = int(np.argmax(table[s])); r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
        if not (0 <= r < D["rows"] and 0 <= c < D["cols"]) or (r, c) in D["walls"]:
            break
        s = (r, c); path.append(s)
        if s == D["goal"] or s in D["hazards"]:
            break
    return path

def draw_domain(D, ax, paths=(), title=""):
    img = np.ones((D["rows"], D["cols"], 3))
    for w in D["walls"]: img[w] = (0.45, 0.45, 0.45)
    for h_ in D["hazards"]: img[h_] = (0.95, 0.55, 0.5)
    img[D["goal"]] = (0.55, 0.85, 0.55); img[D["start"]] = (0.6, 0.75, 0.95)
    ax.imshow(img)
    for p, col in paths:
        ax.plot([c for r, c in p], [r for r, c in p], "-o", color=col, ms=3, lw=2)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)


def reinforce_domain(D, baseline, episodes=1000, lr=0.1, scale=10.0, gamma=0.98, seed=0, max_steps=150):
    """Tabular softmax REINFORCE on the domain, with or without a learned state-value baseline."""
    rng = np.random.default_rng(seed); th = np.zeros((D["rows"], D["cols"], 4)); V = np.zeros((D["rows"], D["cols"])); returns = []
    for _ in range(episodes):
        s = D["start"]; traj = []; G = 0.0
        for _ in range(max_steps):
            p = np.exp(th[s] - th[s].max()); p /= p.sum(); a = int(rng.choice(4, p=p))
            s2, r, done = domain_step(D, s, a, rng); traj.append((s, a, r / scale)); G += r; s = s2
            if done:
                break
        ret = 0.0
        for s_, a_, r_ in reversed(traj):
            ret = r_ + gamma * ret; p = np.exp(th[s_] - th[s_].max()); p /= p.sum()
            adv = ret - V[s_] if baseline else ret
            th[s_] += lr * adv * (np.eye(4)[a_] - p); V[s_] += 0.1 * (ret - V[s_])
        returns.append(G)
    return th, np.array(returns)

def actor_critic_domain(D, episodes=1000, lr_pi=0.3, lr_v=0.2, scale=10.0, gamma=0.98, seed=0, max_steps=150):
    """Tabular one-step actor-critic: the temporal-difference error of the critic is the advantage of the actor."""
    rng = np.random.default_rng(seed); th = np.zeros((D["rows"], D["cols"], 4)); V = np.zeros((D["rows"], D["cols"])); returns = []
    for _ in range(episodes):
        s = D["start"]; G = 0.0
        for _ in range(max_steps):
            p = np.exp(th[s] - th[s].max()); p /= p.sum(); a = int(rng.choice(4, p=p))
            s2, r, done = domain_step(D, s, a, rng); G += r
            delta = r / scale + (0.0 if done else gamma * V[s2]) - V[s]
            th[s] += lr_pi * delta * (np.eye(4)[a] - p); V[s] += lr_v * delta; s = s2
            if done:
                break
        returns.append(G)
    return th, np.array(returns)

dom = make_domain(DOMAIN); pg_runs = {}
for name, fn in (("REINFORCE without baseline", lambda k: reinforce_domain(dom, False, seed=k)),
                 ("REINFORCE with baseline", lambda k: reinforce_domain(dom, True, seed=k)),
                 ("actor-critic", lambda k: actor_critic_domain(dom, seed=k))):
    runs = [fn(k) for k in range(2)]
    pg_runs[name] = (runs[0][0], np.array([r for _, r in runs]))
pg_table = pd.DataFrame({k: [R[:, -300:].mean(), R[:, -300:].std()] for k, (_, R) in pg_runs.items()},
                        index=["mean return, last 300 episodes", "spread of the return"]).T
print(f"{DOMAIN}:"); print(pg_table.round(1).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
for (k, (_, R)), col in zip(pg_runs.items(), ("#c0392b", "#e67e22", "#2980b9")):
    ax[0].plot(pd.Series(R.mean(0)).rolling(50).mean(), color=col, label=k)
ax[0].set_xlabel("episode"); ax[0].set_ylabel("return (moving average)"); ax[0].legend(fontsize=8)
draw_domain(dom, ax[1], [(greedy_path(dom, pg_runs["actor-critic"][0]), "#2980b9")], "most likely path of the actor-critic")
plt.tight_layout(); plt.show()

**Exercise D.** Which learner reaches the highest mean return in your domain? Store its name in `best_learner`.

In [ ]:
best_learner = None   # your answer

check("Exercise D", best_learner, pg_table.iloc[:, 0].idxmax())

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
best_learner = pg_table.iloc[:, 0].idxmax()

check("Exercise D", best_learner, pg_table.iloc[:, 0].idxmax())

## 8. Going further (optional): dense rewards and the learning rate of PPO

*Lecture: [Going further (optional)](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lecture.html#going-further-optional)*

### The same trick on a dense reward

In [ ]:
def dense_step(s, a):
    """Same environment, but partial credit is paid AS the sentence is built."""
    s2 = s + (a,)
    done = is_terminal(s2)
    r = score_sequence(s2) - (score_sequence(s) if len(s) else 0.0)
    return s2, r, done

def dense_rollout(pol, rng):
    s, traj = (), []
    while not is_terminal(s):
        p = pol(s); a = int(rng.choice(NA, p=p))
        s2, r, done = dense_step(s, a)
        traj.append((s, a, r)); s = s2
    return traj

def dense_grad(theta, batch, rng, baseline=None, use_rtg=True):
    P = softmax(theta); pol = lambda s: P[IDX[s]]
    grad = np.zeros_like(theta)
    for _ in range(batch):
        traj = dense_rollout(pol, rng)
        G_total = sum(r for _, _, r in traj)
        G = 0.0
        for t in range(len(traj) - 1, -1, -1):
            s, a, r = traj[t]
            G = r + G
            adv = (G if use_rtg else G_total) - (baseline[IDX[s]] if baseline is not None else 0.0)
            oh = np.zeros(NA); oh[a] = 1.0
            grad[IDX[s]] += adv * (oh - P[IDX[s]])
    return grad / batch

rows = []
for label, kw in (("total return, no baseline", dict(use_rtg=False)),
                  ("reward-to-go", dict(use_rtg=True)),
                  ("reward-to-go + baseline", dict(use_rtg=True, baseline=base))):
    gs = np.array([dense_grad(theta0, 64, np.random.default_rng(200 + i), **kw).ravel()
                   for i in range(60)])
    rows.append(dict(estimator=label, grad_variance=float(gs.var(0).mean())))
dv = pd.DataFrame(rows).set_index("estimator")
print("VARIANCE on the DENSE-reward variant, batch 64\n")
print(dv.round(6).to_string())
print(f"\nreward-to-go vs total return : "
      f"{dv.grad_variance['total return, no baseline'] / dv.grad_variance['reward-to-go']:.2f}x")
print(f"+ baseline                   : "
      f"{dv.grad_variance['total return, no baseline'] / dv.grad_variance['reward-to-go + baseline']:.2f}x")

### How much aggressiveness does the clip absorb?

In [ ]:
# Where does the clip stop helping? Sweep the learning rate for both objectives.
sweep = []
for lr_ in (1.0, 5.0, 20.0, 60.0, 150.0):
    for clip_, nm in ((None, "no clip"), (0.2, "PPO clip 0.2")):
        R = np.array([train_pg(n_iter=200, lr=lr_, clip=clip_, epochs=8,
                               seed=sd)[1]["return"].values for sd in range(3)])
        sweep.append(dict(lr=lr_, config=nm, pct_of_optimum=100 * R[:, -2:].mean() / OPT))
sw = pd.DataFrame(sweep)
print(sw.pivot(index="lr", columns="config", values="pct_of_optimum").round(1).to_string())

fig, ax = plt.subplots(figsize=(6.2, 3.2))
for nm, col in (("no clip", "#c0392b"), ("PPO clip 0.2", "#27ae60")):
    d = sw[sw.config == nm]
    ax.semilogx(d.lr, d.pct_of_optimum, "o-", color=col, label=nm)
ax.set_xlabel("learning rate"); ax.set_ylabel("% of the exact optimum")
ax.set_title("How much aggression can each objective absorb?"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Implement a KL-penalised update instead of the clip and compare it with PPO at the learning rates of section 7. |
| Electronics and Communication Engineering | Use REINFORCE to learn a power-allocation policy over three channels with noisy rewards, and measure the effect of a baseline on the variance. |
| Electrical and Electronics Engineering | Train a softmax policy to choose among four tariff plans for a household battery, with rewards drawn from simulated bills. |
| Mechanical Engineering | Design a two-step decision problem for maintenance timing and compare REINFORCE with an actor-critic on it. |
| Biomedical Engineering | Discuss why an entropy bonus might be required for a treatment-recommendation policy, and how you would measure its entropy fairly. |
| Civil Engineering | Train a policy over three traffic-light timing plans on a simulated intersection and report the spread over ten seeds. |
| Aeronautical Engineering | Repeat the comparison of section 6 with two passes instead of eight and report whether the clip still matters. |
| Biotechnology | Use a softmax policy to choose among four culture media with noisy yields and show the learning curve with and without a baseline over several seeds. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-03/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-03).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>